## Dependencies

In [ ]:
# Connect to Google Drive if the notebook is run on Colab.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Install dependencies if the notebook is run on Colab.
# !pip install emoji mlflow optuna contractions gensim transformers

In [1]:
# Install dependencies if the notebook is run on Kaggle.
from pathlib import Path
if Path('/kaggle/working').exists():
    !pip install mlflow contractions --quiet

In [ ]:
import os
import re
import time
import copy
import html
import json
import random
from datetime import datetime
from collections import Counter, OrderedDict

import emoji
import mlflow
import nltk
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
from contractions import fix
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow import keras
from tensorflow.keras.layers import Dense, Embedding, GRU, LSTM
from tensorflow.keras.metrics import FBetaScore
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tqdm import tqdm
from transformers import DistilBertModel, DistilBertTokenizer

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, fbeta_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

In [ ]:
FLAGS = re.MULTILINE | re.DOTALL

def ensure_nltk_resource(resource_path, download_name=None):
    """Download an NLTK resource only when it is not already available locally."""
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name or resource_path.split('/')[-1], quiet=True)

ensure_nltk_resource('tokenizers/punkt', 'punkt')
ensure_nltk_resource('tokenizers/punkt_tab', 'punkt_tab')
ensure_nltk_resource('corpora/stopwords', 'stopwords')
ensure_nltk_resource('corpora/wordnet', 'wordnet')
ensure_nltk_resource('corpora/omw-1.4', 'omw-1.4')

def resolve_torch_device(min_cuda_capability=(7, 0)):
    """Return the PyTorch device while falling back to CPU when the available GPU is too old
    for the installed Torch build."""
    if not torch.cuda.is_available():
        return 'cpu', None, 'CUDA is not available for PyTorch in this environment.'

    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)
    if gpu_capability < min_cuda_capability:
        message = (
            f'PyTorch will run on CPU because the available GPU ({gpu_name}, sm_{gpu_capability[0]}{gpu_capability[1]}) '
            f'is older than the minimum capability supported by this Torch build (sm_{min_cuda_capability[0]}{min_cuda_capability[1]}).'
        )
        return 'cpu', gpu_name, message

    return 'cuda', gpu_name, 'PyTorch can use the GPU in this environment.'

tf_gpu_available = bool(tf.config.list_physical_devices('GPU'))
device, torch_gpu_name, torch_device_message = resolve_torch_device()

print('TensorFlow version:', tf.__version__)
print('PyTorch version:', torch.__version__)
print('TensorFlow GPU available:', tf_gpu_available)
print('PyTorch CUDA available:', torch.cuda.is_available())
if torch_gpu_name is not None:
    print('Detected GPU for PyTorch:', torch_gpu_name)
print('PyTorch device used by the notebook:', device)
print(torch_device_message)

In [ ]:
RAW_DATA_FILENAME = 'training.1600000.processed.noemoticon.csv'
cwd = Path.cwd().resolve()

# Detect the working environment and declare candidate directories for the raw dataset.
if Path('/kaggle/working').exists():
    RUN_ENV = 'kaggle'
    data_candidates = [
        Path('/kaggle/input/sentiment140'),
        Path('/kaggle/input/datasets/mouhamadoudiagne/sentiment140'),
        Path('/kaggle/input')
    ]
elif Path('/content').exists():
    RUN_ENV = 'colab'
    data_candidates = [
        Path('/content/drive/MyDrive/AI Engineer OC/Projet 7/data/sentiment140'),
        Path('/content/drive/MyDrive/AI Engineer OC/Projet 7'),
        Path('/content')
    ]
else:
    RUN_ENV = 'local'
    data_candidates = [
        cwd / 'data' / 'sentiment140',
        cwd.parent / 'data' / 'sentiment140',
        cwd / 'data',
        cwd.parent / 'data'
    ]

# Expand each candidate so the notebook can work whether the candidate is the dataset root
# itself or a parent directory that contains a `sentiment140` folder.
raw_data_candidates = []
for candidate_dir in data_candidates:
    raw_data_candidates.append(candidate_dir / RAW_DATA_FILENAME)
    raw_data_candidates.append(candidate_dir / 'sentiment140' / RAW_DATA_FILENAME)

RAW_DATA_PATH = next((path for path in raw_data_candidates if path.exists()), None)
if RAW_DATA_PATH is None:
    searched_paths = '\n'.join(f'- {path}' for path in raw_data_candidates)
    raise FileNotFoundError(
        f'Raw dataset not found. The notebook looked for `{RAW_DATA_FILENAME}` in:\n{searched_paths}'
    )

DATA_PATH = RAW_DATA_PATH.parent
print('Working directory :', cwd)
print('Execution environment :', RUN_ENV)
print('Data directory :', DATA_PATH)
print('Raw dataset path :', RAW_DATA_PATH)
print('Raw dataset exists :', RAW_DATA_PATH.exists())